In [1]:
import pandas as pd
import numpy as np

In [2]:
# Path to the raw transcripted CSV (in Data/ folder)
# IMPORTANT: confirm the exact filename matches what's in your Data folder
RAW_PATH = "../Data/data2_done_april1(data2_done_april1).csv"

df = pd.read_csv(RAW_PATH, encoding="latin-1")
print(f"Loaded: {len(df)} rows, {len(df.columns)} columns")
print(f"\nColumns:")
for c in df.columns:
    print(f"  - {c}")

Loaded: 905 rows, 18 columns

Columns:
  - Unnamed: 0
  - Timestamp
  - Your Name
  - Q1: Which account posted this?
  - Q2: Today's date
  - Q3: Which social media platform was this post from?
  - Q4: Post link/URL (in most platforms, click share, and copy URL would do the trick; if impossible to find, please put in NA here.)
  - Full Transcription
  - Q5: Please copy and paste the post's text content/caption here
  - Q6: Please use your own words to briefly describe the content of the image or video, if any (if no image or video, please put in NA)
  - Q7: Your reaction to the post (check all that applies)
  - Q8. Did this feel uncivil/impolite; (Uncivil = rude, insulting, demeaning, threatening, or hostile toward a person or group.)
  - Q9. What MORAL/IMMORAL ideas did the post seem to express? (Select all that apply)
  - Q10. The number of likes/favorites or equivalent engagement of the post.
  - Q11a. The number of comments of the post.
  - Q11b. The number of shares of the post.
 

In [3]:
# 'Your Name' contains the anonymized ID for all rows (e.g., '1_A', '2_A', etc.)
# 'anon_name' is redundant — only filled for first 494 rows. We drop it.
# Also rename Q5 to a workable name.

df = df.rename(columns={"Your Name": "student_id"})
df = df.drop(columns=["anon_name"])

# Rename Q5 to short name
q5_col = [c for c in df.columns if c.startswith("Q5")][0]
df = df.rename(columns={q5_col: "Q5_caption"})

# Verify
print(f"Students: {df['student_id'].nunique()}")
print(f"\nPosts per student:")
print(df['student_id'].value_counts().sort_index())

Students: 12

Posts per student:
student_id
10_A     57
11_A    285
12_A     73
1_A      59
2_A      62
3_A      60
4_A      55
5_A      76
6_A      46
7_A      20
8_A      40
9_A      72
Name: count, dtype: int64


In [4]:
def is_usable_caption(x):
    """Q5 is usable if not NaN, not empty, not literal 'NA'."""
    if pd.isna(x):
        return False
    s = str(x).strip()
    if s == "":
        return False
    if s.upper() == "NA":
        return False
    return True

def is_usable_transcript(x):
    """Transcript is usable if not NaN, not empty, not a 'not a video' marker."""
    if pd.isna(x):
        return False
    s = str(x).strip()
    if s == "":
        return False
    if s.upper() in ["NA", "NOT A VIDEO", "NOT A VIDEO..."]:
        return False
    return True

df["has_caption"] = df["Q5_caption"].apply(is_usable_caption)
df["has_transcript"] = df["Full Transcription"].apply(is_usable_transcript)

print(f"Posts with usable Q5 caption: {df['has_caption'].sum()}")
print(f"Posts with usable transcript: {df['has_transcript'].sum()}")
print(f"Posts with BOTH: {(df['has_caption'] & df['has_transcript']).sum()}")
print(f"Posts with caption only: {(df['has_caption'] & ~df['has_transcript']).sum()}")
print(f"Posts with transcript only: {(~df['has_caption'] & df['has_transcript']).sum()}")
print(f"Posts with NEITHER (will be skipped): {(~df['has_caption'] & ~df['has_transcript']).sum()}")

Posts with usable Q5 caption: 795
Posts with usable transcript: 145
Posts with BOTH: 143
Posts with caption only: 652
Posts with transcript only: 2
Posts with NEITHER (will be skipped): 108


In [5]:
def merge_text(row):
    """Caption first, then transcript. Returns empty string if both missing."""
    cap = str(row["Q5_caption"]).strip() if row["has_caption"] else ""
    tr = str(row["Full Transcription"]).strip() if row["has_transcript"] else ""
    
    if cap and tr:
        return cap + "\n\n" + tr
    elif cap:
        return cap
    elif tr:
        return tr
    else:
        return ""

def get_text_source(row):
    """Label for which input source this row uses."""
    if row["has_caption"] and row["has_transcript"]:
        return "caption+transcript"
    elif row["has_caption"]:
        return "caption_only"
    elif row["has_transcript"]:
        return "transcript_only"
    else:
        return "none"

df["text_for_analysis"] = df.apply(merge_text, axis=1)
df["text_source"] = df.apply(get_text_source, axis=1)
df["has_text"] = df["text_for_analysis"].str.len() > 0

print(f"Total rows with usable text: {df['has_text'].sum()} / {len(df)}")
print(f"\nBreakdown by text source:")
print(df["text_source"].value_counts())

Total rows with usable text: 797 / 905

Breakdown by text source:
text_source
caption_only          652
caption+transcript    143
none                  108
transcript_only         2
Name: count, dtype: int64


In [6]:
for source in ["caption+transcript", "caption_only", "transcript_only"]:
    subset = df[df["text_source"] == source]
    print(f"\n=== {source.upper()} — {len(subset)} posts ===")
    if len(subset) == 0:
        continue
    for _, row in subset.head(2).iterrows():
        text = row["text_for_analysis"]
        print(f"\n  [{row['student_id']}] (length: {len(text)} chars)")
        print(f"  {text[:300]}{'...' if len(text) > 300 else ''}")


=== CAPTION+TRANSCRIPT — 143 posts ===

  [1_A] (length: 1553 chars)
  Chinese authorities say their investigation of a high-profile scandal at one of the countrys leading state-run museums has revealed systemic mismanagement and alleged corruption over decades. #nanjing #museum #china #art #corruption #asia #scmp #scmpnews

It's a high-profile scandal at 1 of China's...

  [1_A] (length: 921 chars)
  You might be feeling overwhelmed with everything thats going on lately
.
Reframing your stress like this is called a stress is enhancing mindset and there is a lot of research that shows your body physically responds better to stress thinking this way????
.
I hope this helps you guys, stay strong ...

=== CAPTION_ONLY — 652 posts ===

  [1_A] (length: 43 chars)
  Super Bowl Monday is a holiday next year ??

  [2_A] (length: 43 chars)
  Explore D.Desirable at lON OrchardSingapore

=== TRANSCRIPT_ONLY — 2 posts ===

  [11_A] (length: 1033 chars)
  Please take a second and switch off the

In [7]:
usable = df[df["has_text"]]
lens = usable["text_for_analysis"].str.len()

print("Text length stats (only rows with text):")
print(f"  Min: {lens.min()}")
print(f"  Max: {lens.max()}")
print(f"  Median: {int(lens.median())}")
print(f"  Mean: {int(lens.mean())}")

print(f"\nText length by source:")
print(usable.groupby("text_source")["text_for_analysis"].apply(lambda x: x.str.len().median()).rename("median chars"))

Text length stats (only rows with text):
  Min: 1
  Max: 32801
  Median: 100
  Mean: 1027

Text length by source:
text_source
caption+transcript    1104.0
caption_only            75.5
transcript_only        554.0
Name: median chars, dtype: float64


In [8]:
usable.head()

,Unnamed: 0,Timestamp,student_id,Q1: Which account posted this?,Q2: Today's date,Q3: Which social media platform was this post from?,"Q4: Post link/URL (in most platforms, click share, and copy URL would do the trick; if impossible to find, please put in NA here.)",Full Transcription,Q5_caption,"Q6: Please use your own words to briefly describe the content of the image or video, if any (if no image or video, please put in NA)",...,Q9. What MORAL/IMMORAL ideas did the post seem to express? (Select all that apply),Q10. The number of likes/favorites or equivalent engagement of the post.,Q11a. The number of comments of the post.,Q11b. The number of shares of the post.,Q12 (optional): Why did this post stand out to you? (1 sentence),has_caption,has_transcript,text_for_analysis,text_source,has_text
0,1.0,08:47.9,1_A,scmpnews,2/11/2026,Instagram,https://www.instagram.com/reel/DUnNzbNCtRj/?ut...,It's a high-profile scandal at 1 of China's le...,Chinese authorities say their investigation of...,A Ming dynasty painting was sold in secret fro...,...,Corruption / Degradation,63.4k,10,NaN,I found it interesting and i've studied ancien...,True,True,Chinese authorities say their investigation of...,caption+transcript,True
1,2.0,16:15.4,1_A,noeldeyzel_bodybuilder,2/11/2026,Instagram,https://www.instagram.com/reel/DUlW8xbkRCm/?ut...,"What's up bro?Sit down, take a breath and list...",You might be feeling overwhelmed with everythi...,He is giving a motivational talk while buildin...,...,"Care (protecting people, reducing harm), Purit...",1.5m,1124,6581 repost,It was a nice motivational post that made me s...,True,True,You might be feeling overwhelmed with everythi...,caption+transcript,True
2,3.0,30:06.0,1_A,ESPN,2/11/2026,Instagram,https://www.instagram.com/p/DUoYHullUIp/?utm_s...,NaN,Super Bowl Monday is a holiday next year ??,A picture of Bad Bunny and next years super bo...,...,NaN,NaN,47,58 reposts,it was funny,True,False,Super Bowl Monday is a holiday next year ??,caption_only,True
3,4.0,56:20.3,2_A,Dylan Wang 1220,2/16/2026,Instagram,NaN,NaN,Explore D.Desirable at lON OrchardSingapore,It's excited to see his own cloth brand opened...,...,"Loyalty (standing with a group, unity), Author...",220000,1647,5047,NaN,True,False,Explore D.Desirable at lON OrchardSingapore,caption_only,True
4,5.0,00:23.7,2_A,idle official,2/16/2026,Instagram,NaN,NaN,i-dle OFFICIAL LIGHT STICK VER.3 COMING SOON,A new light stick.,...,"Authority (respect for rules, leaders, order)",165000,2100,31000,It's very happy to see the new light stick of ...,True,False,i-dle OFFICIAL LIGHT STICK VER.3 COMING SOON,caption_only,True


In [9]:
# Pick the first row that has caption+transcript and print the FULL content
sample = df[df["text_source"] == "caption+transcript"].iloc[0]
print(f"Student: {sample['student_id']}")
print(f"text_source: {sample['text_source']}")
print(f"Total length: {len(sample['text_for_analysis'])} chars")
print()
print("=" * 70)
print("FULL CONTENT:")
print("=" * 70)
print(sample["text_for_analysis"])

Student: 1_A
text_source: caption+transcript
Total length: 1553 chars

FULL CONTENT:
Chinese authorities say their investigation of a high-profile scandal at one of the countrys leading state-run museums has revealed systemic mismanagement and alleged corruption over decades. #nanjing #museum #china #art #corruption #asia #scmp #scmpnews

It's a high-profile scandal at 1 of China's leading state-run cultural institutions Nanjing Museum in eastern Jiangsu province is at the center of a controversy involving national treasures that were funneled to private collectors This Ming Dynasty painting by renowned artist Qiu Ying dates back to the 1500s.It surfaced at an auction in Beijing in early 2025, with an estimated value of 88 million yuan, that's over 12.7 million US dollars.It turns out the painting was among 5 donated works that had been secretly sold in the private art market.The museum's vice director, Xu Huping, has been implicated for allegedly approving the sales.The Qiu Ying pain

In [10]:
output_path = "../Outputs/merged_text_905.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {df.shape}")
print(f"\nColumns saved:")
for c in df.columns:
    print(f"  - {c}")

Saved: ../Outputs/merged_text_905.csv
Shape: (905, 22)

Columns saved:
  - Unnamed: 0
  - Timestamp
  - student_id
  - Q1: Which account posted this?
  - Q2: Today's date
  - Q3: Which social media platform was this post from?
  - Q4: Post link/URL (in most platforms, click share, and copy URL would do the trick; if impossible to find, please put in NA here.)
  - Full Transcription
  - Q5_caption
  - Q6: Please use your own words to briefly describe the content of the image or video, if any (if no image or video, please put in NA)
  - Q7: Your reaction to the post (check all that applies)
  - Q8. Did this feel uncivil/impolite; (Uncivil = rude, insulting, demeaning, threatening, or hostile toward a person or group.)
  - Q9. What MORAL/IMMORAL ideas did the post seem to express? (Select all that apply)
  - Q10. The number of likes/favorites or equivalent engagement of the post.
  - Q11a. The number of comments of the post.
  - Q11b. The number of shares of the post.
  - Q12 (optional): 